# 05 · Binary Data: `bytes`, `bytearray`, `memoryview`, `struct` & Encodings

**Module 11 – Files & OS** · Estimated time: 45 min

> ❗ Not covered in either source, but essential for files, networking, cryptography, images and protocols.

### 🎯 What you will learn
- Text (`str`) vs binary data (`bytes`): encode/decode, encodings, errors
- `bytes` literals, methods, hex, slicing (and why `b[0]` is an `int`)
- Mutable `bytearray`
- Zero-copy views with `memoryview`
- Packing/unpacking binary formats with **`struct`** (endianness!)
- `int.to_bytes` / `from_bytes`, bitwise flags
- Base64/hex encoding, and reading real binary file headers (PNG)

## 1. `str` vs `bytes`

- **`str`**: a sequence of Unicode **characters** (text)
- **`bytes`**: a sequence of **integers 0–255** (raw data: files, network packets, hashes)

Converting between them **always** needs an encoding:
```
"café"  ──encode("utf-8")──►  b'caf\xc3\xa9'  ──decode("utf-8")──►  "café"
```

In [1]:
text = "café ☕"
data = text.encode("utf-8")
print(data, type(data), len(text), "chars →", len(data), "bytes")
print(data.decode("utf-8"))
print(text.encode("utf-16"), len(text.encode("utf-16")))
print(text.encode("ascii", errors="replace"), text.encode("ascii", errors="xmlcharrefreplace"))

b'caf\xc3\xa9 \xe2\x98\x95' <class 'bytes'> 6 chars → 9 bytes
café ☕
b'\xff\xfec\x00a\x00f\x00\xe9\x00 \x00\x15&' 14
b'caf? ?' b'caf&#233; &#9749;'


In [2]:
try:
    data.decode("ascii")
except UnicodeDecodeError as e:
    print("UnicodeDecodeError:", e)
print(data.decode("latin-1"))       # "works", but produces mojibake: the WRONG encoding
print(b"\xff\xfe broken".decode("utf-8", errors="replace"))

UnicodeDecodeError: 'ascii' codec can't decode byte 0xc3 in position 3: ordinal not in range(128)
cafÃ© â
�� broken


> 💡 **Decode bytes into text as early as possible, encode as late as possible** (the "Unicode sandwich"). Inside your program, work with `str`.

## 2. Working with `bytes`

In [3]:
b = b"Hello\x00\xff"                       # literal; \xNN for arbitrary bytes
print(b, len(b))
print(b[0], b[-1])                          # indexing gives INTS
print(b[:5], b[1:2])                        # slicing gives bytes
print(list(b"AB"), bytes([72, 105]))        # ints ↔ bytes
print(b.hex(), b.hex(" "), bytes.fromhex("48 65 6c 6c 6f"))
print(b"a,b,c".split(b","), b"  x ".strip(), b"abc".upper(), b"py" in b"python")
print(bytes(4), b"ab" * 3)                  # zero-filled, repetition

b'Hello\x00\xff' 7
72 255
b'Hello' b'e'
[65, 66] b'Hi'
48656c6c6f00ff 48 65 6c 6c 6f 00 ff b'Hello'
[b'a', b'b', b'c'] b'x' b'ABC' True
b'\x00\x00\x00\x00' b'ababab'


In [4]:
b"abc" + "def"          # ❌ can't mix bytes and str

TypeError: can't concat str to bytes

## 3. `bytearray`: mutable bytes

In [5]:
buf = bytearray(b"hello world")
buf[0] = ord("H")                 # assign an int
buf[6:11] = b"PYTHON"
buf.extend(b"!!")
buf += b"?"
print(buf, bytes(buf))
packet = bytearray()
for chunk in (b"\x01\x02", b"\x03"):
    packet += chunk               # efficient incremental building
print(packet.hex())

bytearray(b'Hello PYTHON!!?') b'Hello PYTHON!!?'
010203


## 4. `memoryview`: zero-copy slices

Slicing `bytes` **copies** data. A `memoryview` slice is a **view** onto the same memory: essential for big buffers and network code.

In [6]:
big = bytearray(b"x" * 10)
view = memoryview(big)
part = view[2:5]                  # no copy
part[:] = b"ABC"                  # writes through to `big`
print(big, part.tobytes(), view.nbytes, view.readonly)

import array
nums = array.array("H", [1, 2, 3])          # unsigned 16-bit ints
mv = memoryview(nums)
print(mv.format, mv.itemsize, mv.tolist(), mv.cast("B").tolist())   # same memory seen as raw bytes

bytearray(b'xxABCxxxxx') b'ABC' 10 False
H 2 [1, 2, 3] [1, 0, 2, 0, 3, 0]


## 5. `struct`: binary formats

`struct.pack(format, values...)` → bytes; `struct.unpack(format, data)` → tuple.

| Code | C type | Size |
|---|---|---|
| `b` / `B` | signed / unsigned char | 1 |
| `h` / `H` | short | 2 |
| `i` / `I` | int | 4 |
| `q` / `Q` | long long | 8 |
| `f` / `d` | float / double | 4 / 8 |
| `?` | bool | 1 |
| `4s` | 4-byte string | 4 |

Byte-order prefix: `<` little-endian, `>` big-endian (= "network order", also `!`), `=` native, no prefix = native with alignment.

In [7]:
import struct

record = struct.pack("<HIf", 7, 123456, 3.5)       # u16, u32, float32, little-endian
print(record, len(record), struct.calcsize("<HIf"))
print(struct.unpack("<HIf", record))

print(struct.pack(">I", 1), struct.pack("<I", 1))  # endianness changes byte order!

# A fixed-size binary record: id (u32), score (f64), name (10 bytes)
Record = struct.Struct("<Id10s")
rows = [Record.pack(1, 91.5, b"Asha"), Record.pack(2, 78.0, b"Ravi")]
blob = b"".join(rows)
for rec_id, score, name in Record.iter_unpack(blob):
    print(rec_id, score, name.rstrip(b"\x00").decode())

b'\x07\x00@\xe2\x01\x00\x00\x00`@' 10 10
(7, 123456, 3.5)
b'\x00\x00\x00\x01' b'\x01\x00\x00\x00'
1 91.5 Asha
2 78.0 Ravi


## 6. Integers ↔ bytes, and bit flags

In [8]:
n = 1025
print(n.to_bytes(2, "big"), n.to_bytes(4, "little"), int.from_bytes(b"\x04\x01", "big"))
print((-1).to_bytes(2, "big", signed=True))

READ, WRITE, EXEC = 0b100, 0b010, 0b001
perms = READ | WRITE
print(f"{perms:03b}", bool(perms & WRITE), bool(perms & EXEC))
perms &= ~WRITE                                  # clear a bit
print(f"{perms:03b}")

b'\x04\x01' b'\x01\x04\x00\x00' 1025
b'\xff\xff'
110 True False
100


## 7. Base64 and hex: binary as text

In [9]:
import base64, binascii

raw = b"\x89PNG\r\n binary \x00\xff"
b64 = base64.b64encode(raw)
print(b64, base64.b64decode(b64) == raw)
print(base64.urlsafe_b64encode(b"any bytes?/+"))     # safe in URLs / JWTs
print(binascii.hexlify(raw), raw.hex())

b'iVBORw0KIGJpbmFyeSAA/w==' True
b'YW55IGJ5dGVzPy8r'
b'89504e470d0a2062696e6172792000ff' 89504e470d0a2062696e6172792000ff


## 8. Real example: reading a PNG header

PNG files start with an 8-byte signature, then an `IHDR` chunk holding width and height as big-endian 32-bit ints:

In [10]:
with open("blue_color.png", "rb") as f:          # an image from the course assets
    header = f.read(24)
print(header[:8])
print("is PNG:", header[:8] == b"\x89PNG\r\n\x1a\n")
length, chunk_type = struct.unpack(">I4s", header[8:16])
width, height = struct.unpack(">II", header[16:24])
print(chunk_type, f"{width}x{height}")

b'\x89PNG\r\n\x1a\n'
is PNG: True
b'IHDR' 200x200


The same techniques read WAV/ZIP/BMP headers, network protocols (`socket.recv` returns bytes), and binary file formats.

## 📝 Summary
- `str` is text, `bytes` is data; convert with `.encode()`/`.decode()` and an explicit encoding.
- `bytes` is immutable (indexing gives ints); `bytearray` is mutable; `memoryview` gives zero-copy views.
- `struct` packs/unpacks binary layouts; mind the byte order (`<`, `>`).
- `int.to_bytes`/`from_bytes`, bit flags, base64 and hex round out binary handling.

## 🏋️ Practice
1. Write `is_jpeg(path)` (JPEG files start with `FF D8 FF`).
2. Pack a list of `(x, y)` float pairs into bytes with `struct` and unpack them again.
3. Write a function that XOR-"encrypts" bytes with a repeating key (and decrypts with the same function).
4. Count how many bytes a string needs in UTF-8 vs UTF-16 vs UTF-32 for English, Hindi and emoji text.

<details><summary>Show solutions</summary>

```python
import struct
from itertools import cycle

def is_jpeg(path):
    with open(path, "rb") as f:
        return f.read(3) == b"\xff\xd8\xff"

pts = [(1.0, 2.0), (3.5, -4.25)]
blob = b"".join(struct.pack("<dd", *p) for p in pts)
print([p for p in struct.iter_unpack("<dd", blob)])

def xor(data: bytes, key: bytes) -> bytes:
    return bytes(b ^ k for b, k in zip(data, cycle(key)))
secret = xor(b"hello", b"key")
print(secret, xor(secret, b"key"))

for s in ["hello", "नमस्ते", "😀"]:
    print(s, [len(s.encode(e)) for e in ("utf-8", "utf-16-le", "utf-32-le")])
```
</details>